In [123]:
# --- Package import ---

import pandas as pd
import folium
from folium import plugins
import numpy as np
from scipy.spatial.distance import cdist

import calliope

from ruamel.yaml import YAML
import pyrosm

import geopandas as gpd
from shapely.geometry import Point, LineString

from owslib.wfs import WebFeatureService
import io

import requests
import os

In [124]:
# --- Haversine distance function ---

def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance in kilometers between two points 
    on the earth (specified in decimal degrees).
    """
    # Convert decimal degrees to radians 
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    # Haversine formula 
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a)) 
    # Radius of earth in kilometers. Use 3956 for miles
    r = 6371 
    return c * r

In [125]:
# --- API calls to obtain geodata ---

# Fetch building GeoJSON data for area 4011 from the TNO Warmteprofielgenerator API (may not be legal)
geojson_url = "https://hlc-api.warmteprofielengenerator.nl/building_data/geojson/4011"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:145.0) Gecko/20100101 Firefox/145.0",
    "Accept": "application/json",
    "Referer": "https://www.warmteprofielengenerator.nl/",
    "Origin": "https://www.warmteprofielengenerator.nl"
}

response = requests.get(geojson_url, headers=headers)
response.raise_for_status()
data = response.json()

# Extract properties from each feature
if "features" in data and len(data["features"]) > 0:
    properties_list = [f.get("properties", {}) for f in data["features"]]
    residential_heat_demand = pd.DataFrame(properties_list)
else:
    print("No features to export.")

# Extract only the heat demand data from the InfluxDB/Grafana API
url = (
    "https://hlc-grafana.warmteprofielengenerator.nl/api/datasources/proxy/2/query?db=tnohlc"
    "&q=SELECT%20sum(%22P_heat%22)%20FROM%20%22tot-4011%22%20WHERE%20time%20%3E%3D%201546300800000ms%20and%20time%20%3C%3D%201577836799999ms%20GROUP%20BY%20time(1h)%20fill(null)"
    "&epoch=ms" )

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:145.0) Gecko/20100101 Firefox/145.0",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "Referer": "https://hlc-grafana.warmteprofielengenerator.nl/d-solo/BJBoKWivz/warmtevraagprofiel-2019-4011?orgId=1&panelId=1&from=1546300800000&to=1577836799999&theme=light",
    "x-grafana-org-id": "1",
    "DNT": "1",
    "Connection": "keep-alive",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-origin",
    "Priority": "u=4",
    "TE": "trailers"
}

response = requests.get(url, headers=headers)
response.raise_for_status()
data = response.json()

# Query BAG buildings (pand) from PDOK WFS API as GeoJSON

# Define your bounding box (minx, miny, maxx, maxy) in EPSG:4326 (WGS84)
bbox = [4.354481303046308, 51.990211796688996, 4.36349681889779, 51.99831039946793]

# WFS API returns maxFeatures=1000 by default; we need to paginate to get more
def fetch_all_features(bbox, max_features=1000):
    features = []
    start_index = 0
    while True:
        wfs_url = (
            "https://service.pdok.nl/lv/bag/wfs/v2_0?"
            "service=WFS&version=1.1.0&request=GetFeature"
            "&typeName=bag:pand"
            "&outputFormat=application/json"
            f"&bbox={bbox[1]},{bbox[0]},{bbox[3]},{bbox[2]},urn:ogc:def:crs:EPSG::4326"
            f"&maxFeatures={max_features}"
            f"&startIndex={start_index}"
        )
        response = requests.get(wfs_url)
        response.raise_for_status()
        gdf = gpd.read_file(io.BytesIO(response.content))
        if gdf.empty:
            break
        features.append(gdf)
        if len(gdf) < max_features:
            break
        start_index += max_features
    if features:
        return gpd.GeoDataFrame(pd.concat(features, ignore_index=True), crs=features[0].crs)
    else:
        return gpd.GeoDataFrame()

gdf = fetch_all_features(bbox)

In [126]:
# --- Demand node definition ---

# Extract the heat demand series only
series = data.get("results", [])[0].get("series", []) if "results" in data else []
heat_df = None
for s in series:
    name = s.get("name", "")
    columns = s.get("columns", [])
    values = s.get("values", [])
    if name == "tot-4011":
        heat_df = pd.DataFrame(values, columns=columns)

# --- New calculation as requested ---
if (residential_heat_demand is not None) and (heat_df is not None):
    warmtevraag_sum = residential_heat_demand["Warmtevraag"].sum()
    # Calculate share for each building
    residential_heat_demand["share"] = residential_heat_demand["Warmtevraag"] / warmtevraag_sum
    # Get max hourly heat demand from heat_df
    max_heat = heat_df["sum"].max()
    # Multiply share by max_heat
    residential_heat_demand["Peak heat demand (kW)"] = residential_heat_demand["share"] * max_heat/1000
else:
    print("Data not available for calculation.")

# Reproject to a projected CRS for centroid calculation, then convert centroids back to WGS84 for lon/lat
if gdf.crs is not None and gdf.crs.to_epsg() == 4326:
    gdf_proj = gdf.to_crs(epsg=28992)  # Projected CRS for NL
    centroids_proj = gdf_proj.centroid
    centroids_wgs = gpd.GeoSeries(centroids_proj, crs=28992).to_crs(epsg=4326)
    gdf['lon'] = centroids_wgs.x
    gdf['lat'] = centroids_wgs.y
else:
    # Fallback: use centroid for non-point geometries, but ensure output is in WGS84
    centroids = gdf.geometry.centroid
    centroids_wgs = gpd.GeoSeries(centroids, crs=gdf.crs).to_crs(epsg=4326)
    gdf['lon'] = centroids_wgs.x
    gdf['lat'] = centroids_wgs.y

# Prepare export DataFrame with identification and coordinates
export_cols = ['identificatie', 'lon', 'lat']
export_df = gdf[export_cols].copy() if all(col in gdf.columns for col in export_cols) else gdf[[c for c in export_cols if c in gdf.columns]].copy()
export_df = export_df.rename(columns={'identificatie': 'id'})

# Merge residential_heat_demand with export_df on 'id'
merged_df = pd.merge(residential_heat_demand, export_df, on='id', how='inner')

In [127]:
# --- Transmission node definition ---

import warnings
warnings.filterwarnings("ignore", message=".*ChainedAssignmentError: behaviour will change in pandas 3.0!.*")

# Path to the PBF file
pbf_path = "inputs/delft.osm.pbf"

# Define the bounding box as a shapely Polygon with four vertices (WGS84 coordinates)
from shapely.geometry import Polygon

# List of (lon, lat) tuples for the four corners (in order, must form a closed ring)
bbox_coords = [
    (4.358862898190234, 51.989950476011565),
    (4.363513483963136, 51.99116041768696),
    (4.359959662710779, 51.997215138653104),
    (4.356868839817735, 51.996580034452485),
    (4.355168730042115, 51.995518297847504),
    (4.35819822166681, 51.9901601698577),
    (4.358862898190234, 51.989950476011565)    # repeat the first point to close the polygon
]
bbox_polygon = Polygon(bbox_coords)

# Load the OSM data with bounding polygon
osm = pyrosm.OSM(pbf_path, bounding_box=bbox_polygon)

# Get all roads and foot paths (including footways, cycleways, etc.)
streets = osm.get_network(network_type="all")

# Ensure geometries are LineStrings or MultiLineStrings
streets = streets[streets.geometry.type.isin(["LineString", "MultiLineString"])]

# Explode MultiLineStrings to LineStrings
streets_exploded = streets.explode(index_parts=False).reset_index(drop=True)

# Collect all start and end points as shapely Points
all_points = []
for geom in streets_exploded.geometry:
    all_points.append(Point(geom.coords[0]))
    all_points.append(Point(geom.coords[-1]))

all_points_gs = gpd.GeoSeries(all_points, crs=streets_exploded.crs)

# Count occurrences of each point (intersections occur more than once)
points_df = all_points_gs.value_counts().reset_index()
points_df.columns = ["geometry", "count"]

# Identify cul-de-sacs (endpoints that appear only once)
culdesacs_df = points_df[points_df["count"] == 1].copy()
culdesacs_df = culdesacs_df.copy()
culdesacs_df.loc[:, "type"] = "culdesac"

# Identify intersections (points that appear more than once)
intersections_df = points_df[points_df["count"] > 1].copy()
intersections_df = intersections_df.copy()
intersections_df.loc[:, "type"] = "intersection"

# Combine intersections and cul-de-sacs
all_nodes_df = pd.concat([intersections_df, culdesacs_df], ignore_index=True)

# Convert to GeoDataFrame
all_nodes_gdf = gpd.GeoDataFrame(all_nodes_df, geometry="geometry", crs=streets_exploded.crs)

# Project to WGS84 if needed
if all_nodes_gdf.crs != "EPSG:4326":
    all_nodes_gdf = all_nodes_gdf.to_crs(epsg=4326)

all_nodes_gdf = all_nodes_gdf.copy()
all_nodes_gdf.loc[:, "lon"] = all_nodes_gdf.geometry.x
all_nodes_gdf.loc[:, "lat"] = all_nodes_gdf.geometry.y

In [128]:
# --- Interpolate between transmission nodes (all_nodes_gdf) ---

# Set the desired spacing in meters between interpolated nodes
spacing_m = 5  # Change this value to control node density

# Use the haversine_distance function already defined in the notebook
# (returns distance in kilometers)
def interpolate_line(lat1, lon1, lat2, lon2, spacing_m=1.0):
    total_dist_km = haversine_distance(lat1, lon1, lat2, lon2)
    total_dist_m = total_dist_km * 1000
    if total_dist_m < 1e-6:
        return [(lat1, lon1), (lat2, lon2)]
    n_points = int(np.floor(total_dist_m / spacing_m))
    if n_points < 1:
        return [(lat1, lon1), (lat2, lon2)]
    lats = np.linspace(lat1, lat2, n_points + 2)
    lons = np.linspace(lon1, lon2, n_points + 2)
    return list(zip(lats, lons))

# Build a list of all unique node pairs (edges) from the street network
edges = []
for idx, street in streets_exploded.iterrows():
    coords = list(street.geometry.coords)
    for i in range(len(coords) - 1):
        pt1 = coords[i]
        pt2 = coords[i+1]
        edges.append((pt1, pt2))

# Map node coordinates to their (rounded) values for matching
node_coords_set = set([(round(pt.x, 6), round(pt.y, 6)) for pt in all_nodes_gdf.geometry])

# Only keep edges where both endpoints are in the node set
filtered_edges = [e for e in edges if (round(e[0][0], 6), round(e[0][1], 6)) in node_coords_set and (round(e[1][0], 6), round(e[1][1], 6)) in node_coords_set]

# Interpolate points for each edge using the global spacing_m
interpolated_points = []
for pt1, pt2 in filtered_edges:
    lat1, lon1 = pt1[1], pt1[0]
    lat2, lon2 = pt2[1], pt2[0]
    points = interpolate_line(lat1, lon1, lat2, lon2, spacing_m=spacing_m)
    interpolated_points.extend(points)

# Remove duplicates and create a GeoDataFrame
unique_points = list({(round(lat, 7), round(lon, 7)) for lat, lon in interpolated_points})
interp_gdf = gpd.GeoDataFrame(geometry=[Point(lon, lat) for lat, lon in unique_points], crs="EPSG:4326")
interp_gdf["lat"] = interp_gdf.geometry.y
interp_gdf["lon"] = interp_gdf.geometry.x

In [129]:
# --- Folium map: interpolated transmission nodes and demand nodes ---

demand_nodes=merged_df
transmission_nodes=interp_gdf

# Center the map on the mean coordinates of all points (demand + interpolated transmission)
center_lat = np.mean(np.concatenate([demand_nodes['lat'].values, interp_gdf['lat'].values]))
center_lon = np.mean(np.concatenate([demand_nodes['lon'].values, interp_gdf['lon'].values]))

map2 = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles="OpenStreetMap")

# Add demand nodes (red)
for _, row in demand_nodes.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=3,
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.7,
        popup=folium.Popup(f"<b>Demand Node</b><br>ID: {row.get('id', row.get('nodes', ''))}", max_width=250)
    ).add_to(map2)

# Add interpolated transmission nodes (blue)
for _, row in interp_gdf.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=2,
        color='blue',
        fill=True,
        fill_color='blue',
        fill_opacity=0.5,
        popup=folium.Popup(f"<b>Transmission Node</b><br>Lat: {row['lat']:.6f}<br>Lon: {row['lon']:.6f}", max_width=200)
    ).add_to(map2)

# --- Add lines between interpolated nodes along each street segment ---
# For each street segment, interpolate points and connect each consecutive pair
for idx, street in streets_exploded.iterrows():
    coords = list(street.geometry.coords)
    for i in range(len(coords) - 1):
        pt1 = coords[i]
        pt2 = coords[i+1]
        # Interpolate points for this segment using the same function and spacing_m
        lat1, lon1 = pt1[1], pt1[0]
        lat2, lon2 = pt2[1], pt2[0]
        points = interpolate_line(lat1, lon1, lat2, lon2, spacing_m=spacing_m)
        # Draw lines between each consecutive pair of interpolated points
        for j in range(len(points) - 1):
            start_lat, start_lon = points[j]
            end_lat, end_lon = points[j+1]
            folium.PolyLine(
                locations=[[start_lat, start_lon], [end_lat, end_lon]],
                color="#3186cc",
                weight=2,
                opacity=1
            ).add_to(map2)

# Save and preview
map2.save("intermediary_map.html")

In [130]:
# --- Import csv files from inputs folder

inputs_folder = "inputs"

# Dictionary to hold DataFrames
input_dataframes = {}

# Read all CSV files in the folder
for filename in os.listdir(inputs_folder):
    if filename.endswith(".csv"):
        # Create a descriptive variable name from the filename
        var_name = filename.replace(".csv", "").replace("-", "_").replace(" ", "_")
        df = pd.read_csv(os.path.join(inputs_folder, filename))
        input_dataframes[var_name] = df
        globals()[var_name] = df  # Optionally set as a variable in the global namespace

In [ ]:
# 1. Combine all_nodes_gdf and interp_gdf into a single DataFrame of nodes
all_points_df = pd.concat([all_nodes_gdf[['lon', 'lat']], interp_gdf[['lon', 'lat']]], ignore_index=True).drop_duplicates()
all_points_df = all_points_df.reset_index(drop=True)
all_points_df['id'] = ['transmission%d' % i for i in range(len(all_points_df))]

# 2. Build a mapping from (lon, lat) to node ID
coord_to_id = {(round(row.lon, 7), round(row.lat, 7)): row.id for _, row in all_points_df.iterrows()}

# 3. For each street segment, interpolate points and create links between consecutive points
links = []
for _, street in streets_exploded.iterrows():
    coords = list(street.geometry.coords)
    for i in range(len(coords) - 1):
        lat1, lon1 = coords[i][1], coords[i][0]
        lat2, lon2 = coords[i+1][1], coords[i+1][0]
        points = interpolate_line(lat1, lon1, lat2, lon2, spacing_m=spacing_m)
        # For each consecutive pair of points, create links
        for j in range(len(points) - 1):
            pt1 = (round(points[j][1], 7), round(points[j][0], 7))
            pt2 = (round(points[j+1][1], 7), round(points[j+1][0], 7))
            if pt1 in coord_to_id and pt2 in coord_to_id:
                node_from = coord_to_id[pt1]
                node_to = coord_to_id[pt2]
                # Heat link
                links.append({
                    "techs": f"{node_from}_to_{node_to}_heat",
                    "color": "#823740",
                    "name": "LQ heat distribution",
                    "base_tech": "transmission",
                    "flow_cap_max": 10000,
                    "flow_out_eff_per_distance": 0.99,
                    "lifetime": 20,
                    "link_to": node_to,
                    "link_from": node_from
                })
                # Electricity link
                links.append({
                    "techs": f"{node_from}_to_{node_to}_electricity",
                    "color": "#3186cc",
                    "name": "LV electricity distribution",
                    "base_tech": "transmission",
                    "flow_cap_max": 10000,
                    "flow_out_eff_per_distance": 0.99,
                    "lifetime": 20,
                    "link_to": node_to,
                    "link_from": node_from
                })

# 4. Convert links to DataFrame
links_techs = pd.DataFrame(links)

In [ ]:
# --- Add automatically generated nodes to dataframes and create links

transmission_nodes = transmission_nodes.copy()
transmission_nodes["id"] = [f"transmission{i+1}" for i in range(len(transmission_nodes))]

# Update every value in the 'id' column to 'D_' + original value
demand_nodes['id'] = 'D' + demand_nodes['id'].astype(str)

demand_techs = pd.DataFrame({
    "nodes": demand_nodes["id"],
    "techs": "demand_LQ_heat",
    "parameters": "sink_use_equals",
    "timesteps": "",
    "2050/01/01 00:00": demand_nodes["Peak heat demand (kW)"]
})

# 2. Prepare transmission node tech assignments (value = 0)
transmission_techs = pd.DataFrame({
    "nodes": transmission_nodes["id"],
    "techs": "demand_LQ_heat",
    "parameters": "sink_use_equals",
    "timesteps": "",
    "2050/01/01 00:00": 0
})

# 3. Concatenate both and append to warmtenet_nodes_techs
new_techs = pd.concat([demand_techs, transmission_techs], ignore_index=True)
old_techs = pd.concat([warmtenet_nodes_techs, MV_LV_transformer_nodes_techs], ignore_index=True)
nodes_techs = pd.concat([old_techs, new_techs], ignore_index=True)

# Prepare DataFrames for demand and transmission node coordinates
demand_coords = demand_nodes[["id", "lon", "lat"]].copy().rename(columns={"id": "nodes", "lon": "longitude", "lat": "latitude"})
transmission_coords = transmission_nodes[["id", "lon", "lat"]].copy().rename(columns={"id": "nodes", "lon": "longitude", "lat": "latitude"})

# Concatenate both
new_coords = pd.concat([demand_coords, transmission_coords], ignore_index=True)
old_coords = pd.concat([warmtenet_nodes_coordinates, MV_LV_transformer_nodes_coordinates], ignore_index=True)
nodes_coordinates= pd.concat([old_coords, new_coords], ignore_index=True)

# Build a mapping from coordinates to transmission node IDs
coord_to_id = {(round(row.lon, 7), round(row.lat, 7)): row.id for _, row in transmission_nodes.iterrows()}

# Collect links between connected transmission nodes
links = []
for _, street in streets_exploded.iterrows():
    coords = list(street.geometry.coords)
    for i in range(len(coords) - 1):
        pt1 = (round(coords[i][0], 7), round(coords[i][1], 7))
        pt2 = (round(coords[i+1][0], 7), round(coords[i+1][1], 7))
        # Only add link if both endpoints are transmission nodes
        if pt1 in coord_to_id and pt2 in coord_to_id:
            node_from = coord_to_id[pt1]
            node_to = coord_to_id[pt2]
            # Heat transmission link
            link_name_heat = f"{node_from}_to_{node_to}_heat"
            links.append({
                "techs": link_name_heat,
                "color": "#823740",
                "name": "LQ heat distribution",
                "base_tech": "transmission",
                "flow_cap_max": 10000,
                "flow_out_eff_per_distance": 0.99,
                "lifetime": 20,
                "link_to": node_to,
                "link_from": node_from
            })
            # Electricity transmission link
            link_name_elec = f"{node_from}_to_{node_to}_electricity"
            links.append({
                "techs": link_name_elec,
                "color": "#3186cc",
                "name": "LV electricity distribution",
                "base_tech": "transmission",
                "flow_cap_max": 10000,
                "flow_out_eff_per_distance": 0.99,
                "lifetime": 20,
                "link_to": node_to,
                "link_from": node_from
            })

# Create DataFrame
new_links = pd.DataFrame(links)

links_techs= pd.concat([warmtenet_links_techs, new_links], ignore_index=True)

In [132]:
# --- Connect demand nodes, substations, and transformers to distribution network

# 1. Get coordinates of substation_multatulibuurt
substation_row = warmtenet_nodes_coordinates[warmtenet_nodes_coordinates['nodes'] == 'substation_multatulibuurt']
if substation_row.empty:
    raise ValueError("substation_multatulibuurt not found in warmtenet_nodes_coordinates")
sub_lon = substation_row.iloc[0]['longitude']
sub_lat = substation_row.iloc[0]['latitude']

# 2. Get transmission nodes and their coordinates
trans_nodes = transmission_nodes[['id', 'lon', 'lat']].copy()

# 3. Find nearest transmission node using haversine_distance
min_dist = float('inf')
nearest_trans_id = None
for _, row in trans_nodes.iterrows():
    dist = haversine_distance(sub_lat, sub_lon, row['lat'], row['lon'])
    if dist < min_dist:
        min_dist = dist
        nearest_trans_id = row['id']

# 4. Create the heat link dictionary
link_name = f"substation_multatulibuurt_to_{nearest_trans_id}_heat"
new_link = {
    "techs": link_name,
    "color": "#823740",
    "name": "LQ heat distribution",
    "base_tech": "transmission",
    "flow_cap_max": 10000,
    "flow_out_eff_per_distance": 0.99,
    "lifetime": 20,
    "link_to": nearest_trans_id,
    "link_from": "substation_multatulibuurt"
}

# 5. Append to links_techs DataFrame (use pd.concat instead of append)
links_techs = pd.concat([links_techs, pd.DataFrame([new_link])], ignore_index=True)

# For each MV/LV transformer node, connect to nearest transmission node with an electricity link
new_elec_links = []
for _, mv_row in MV_LV_transformer_nodes_coordinates.iterrows():
    mv_node = mv_row['nodes']
    mv_lon = mv_row['longitude']
    mv_lat = mv_row['latitude']
    # Find nearest transmission node
    min_dist = float('inf')
    nearest_trans_id = None
    for _, trans_row in transmission_nodes.iterrows():
        dist = haversine_distance(mv_lat, mv_lon, trans_row['lat'], trans_row['lon'])
        if dist < min_dist:
            min_dist = dist
            nearest_trans_id = trans_row['id']
    # Create electricity link
    link_name = f"{mv_node}_to_{nearest_trans_id}_electricity"
    new_link = {
        "techs": link_name,
        "color": "#3186cc",
        "name": "LV electricity distribution",
        "base_tech": "transmission",
        "flow_cap_max": 10000,
        "flow_out_eff_per_distance": 0.99,
        "lifetime": 20,
        "link_to": nearest_trans_id,
        "link_from": mv_node
    }
    new_elec_links.append(new_link)

# Append all new links to links_techs DataFrame
links_techs = pd.concat([links_techs, pd.DataFrame(new_elec_links)], ignore_index=True)

# For each demand node, connect to nearest transmission node with both heat and electricity links
new_demand_links = []
for _, demand_row in demand_nodes.iterrows():
    demand_node = demand_row['id']
    demand_lon = demand_row['lon']
    demand_lat = demand_row['lat']
    # Find nearest transmission node
    min_dist = float('inf')
    nearest_trans_id = None
    for _, trans_row in transmission_nodes.iterrows():
        dist = haversine_distance(demand_lat, demand_lon, trans_row['lat'], trans_row['lon'])
        if dist < min_dist:
            min_dist = dist
            nearest_trans_id = trans_row['id']
    # Create heat link
    link_name_heat = f"{demand_node}_to_{nearest_trans_id}_heat"
    new_link_heat = {
        "techs": link_name_heat,
        "color": "#823740",
        "name": "LQ heat distribution",
        "base_tech": "transmission",
        "flow_cap_max": 10000,
        "flow_out_eff_per_distance": 0.99,
        "lifetime": 20,
        "link_to": nearest_trans_id,
        "link_from": demand_node
    }
    # Create electricity link
    link_name_elec = f"{demand_node}_to_{nearest_trans_id}_electricity"
    new_link_elec = {
        "techs": link_name_elec,
        "color": "#3186cc",
        "name": "LV electricity distribution",
        "base_tech": "transmission",
        "flow_cap_max": 10000,
        "flow_out_eff_per_distance": 0.99,
        "lifetime": 20,
        "link_to": nearest_trans_id,
        "link_from": demand_node
    }
    new_demand_links.extend([new_link_heat, new_link_elec])

# Append all new links to links_techs DataFrame
links_techs = pd.concat([links_techs, pd.DataFrame(new_demand_links)], ignore_index=True)

In [133]:
# Create an empty copy of warmtenet_links_carriers with the same columns and dtypes
links_LQ_heat = warmtenet_links_carriers.iloc[0:0].copy()

# Select all 'techs' values from links_techs where 'name' is "LQ heat distribution"
lq_heat_techs = links_techs.loc[links_techs['name'] == "LQ heat distribution", 'techs']

# Create a DataFrame with these techs and fill all other columns with 1
new_rows = pd.DataFrame(1, index=range(len(lq_heat_techs)), columns=links_LQ_heat.columns)
new_rows['techs'] = lq_heat_techs.values

# Concatenate to links_LQ_heat
links_LQ_heat = pd.concat([links_LQ_heat, new_rows], ignore_index=True)

links_LQ_heat.to_csv('links_LQ_heat.csv', index=False)

# Select all 'techs' values from links_techs where 'name' is "LV electricity distribution"
lv_elec_techs = links_techs.loc[links_techs['name'] == "LV electricity distribution", 'techs']

# Create a DataFrame with these techs and fill all other columns with 1
new_rows_elec = pd.DataFrame(1, index=range(len(lv_elec_techs)), columns=links_LQ_heat.columns)
new_rows_elec['techs'] = lv_elec_techs.values

# Concatenate to a new DataFrame, e.g., links_LV_electricity
links_electricity = links_LQ_heat.iloc[0:0].copy()
links_electricity = pd.concat([links_electricity, new_rows_elec], ignore_index=True)

# Create a DataFrame with 'techs' from links_techs and a constant value in 'cost_flow_cap_per_distance'
links_costs = pd.DataFrame({
    'techs': links_techs['techs'],
    'cost_flow_cap_per_distance': 100
})

In [134]:
# --- Save dataframes to csv files
warmtenet_links_carriers.to_csv('data_tables/warmtenet_links_carriers.csv', index=False)
nodes_techs.to_csv('data_tables/nodes_techs.csv', index=False)
nodes_coordinates.to_csv('data_tables/nodes_coordinates.csv', index=False)
links_techs.to_csv('data_tables/links_techs.csv', index=False)
links_LQ_heat.to_csv('data_tables/links_LQ_heat.csv', index=False)
links_electricity.to_csv('data_tables/links_electricity.csv', index=False)
links_costs.to_csv('data_tables/links_costs.csv', index=False)

In [135]:


# 1. Prepare node coordinates DataFrame
# nodes_coordinates: columns should include 'nodes', 'latitude', 'longitude'
node_coords = nodes_coordinates.set_index('nodes')

# 2. Prepare links DataFrame
# links_techs: columns should include 'link_from', 'link_to', and optionally 'techs', 'name', etc.
# If 'link_from' and 'link_to' are not present, extract them from 'techs' (format: node_from_to_node_to_type)
if 'link_from' not in links_techs.columns or 'link_to' not in links_techs.columns:
    links_techs = links_techs.copy()
    links_techs[['link_from', 'link_to']] = links_techs['techs'].str.extract(r'^(.*?)_to_(.*?)_')

# 3. Create a Folium map centered on the mean coordinates
center_lat = node_coords['latitude'].mean()
center_lon = node_coords['longitude'].mean()
network_map = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles='OpenStreetMap')

# 4. Add nodes as circle markers
for node, row in node_coords.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=4,
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.7,
        popup=f"Node: {node}"
    ).add_to(network_map)

# 5. Add links as lines
for _, link in links_techs.iterrows():
    from_node = link['link_from']
    to_node = link['link_to']
    if from_node in node_coords.index and to_node in node_coords.index:
        from_lat, from_lon = node_coords.loc[from_node, ['latitude', 'longitude']]
        to_lat, to_lon = node_coords.loc[to_node, ['latitude', 'longitude']]
        folium.PolyLine(
            locations=[[from_lat, from_lon], [to_lat, to_lon]],
            color='blue',
            weight=2,
            opacity=0.6,
            popup=f"{from_node} → {to_node}"
        ).add_to(network_map)

# 6. Save and display the map
network_map.save('outputs/network_map.html')

In [140]:
# --- Scenario creation and model running ---

#calliope.set_log_verbosity("INFO", include_solver_output=True)

# --- Scenario definition ---
# Set the scenario to 'full_electrification' or 'district heating'.
scenario = 'district_heating'

if scenario == 'full_electrification':
    
    # 1. Load the nodes data that was just created in the previous cell
    nodes_base_info_df = pd.read_csv('data_tables/nodes_base_info.csv')
    demand_nodes = nodes_base_info_df[nodes_base_info_df['nodes'].str.startswith('D')]['nodes']

    # 2. Read the model.yaml file
    # Using ruamel.yaml to preserve comments and structure
    yaml = YAML()
    yaml_path = 'district_heating_model.yaml'
    with open(yaml_path, 'r') as f:
        model_config = yaml.load(f)


    
    # 3. Add the heat pump technology to each demand node in the YAML structure
    # Create the top-level 'nodes' key if it doesn't exist
    if 'nodes' not in model_config:
        model_config['nodes'] = {}
    
    # For each demand node, add an entry to allow 'heat_pump' to be built
    for node_name in demand_nodes:
        if node_name not in model_config['nodes']:
            model_config['nodes'][node_name] = {}
        if 'techs' not in model_config['nodes'][node_name]:
            model_config['nodes'][node_name]['techs'] = {}
        # Adding the technology with an empty dictionary is enough to make it available
        model_config['nodes'][node_name]['techs']['heat_pump'] = {}
        
    # 4. Write the updated configuration back to the model.yaml file
    new_yaml_path = 'electrification_model.yaml'
    with open(new_yaml_path, 'w') as f:
        yaml.dump(model_config, f)

    # 5. Deactivate district heating supply node
    nodes_df = pd.read_csv('data_tables/nodes.csv')
    sh_demand_electricity_mask = (nodes_df['nodes'].str.startswith('SH'))
    nodes_df.loc[sh_demand_electricity_mask, '2050/01/01 00:00'] = 0
    nodes_df.to_csv('data_tables/nodes.csv', index=False)
    
    model = calliope.read_yaml("electrification_model.yaml")

elif scenario == 'district_heating':
    model = calliope.read_yaml("district_heating_model.yaml")


In [141]:
# --- Building and solving of Calliope model ---

model.build()
model.solve()

In [142]:
model.results

<xarray.Dataset> Size: 503MB
Dimensions:                     (nodes: 1195, techs: 2023, carriers: 3,
                                 timesteps: 1, costs: 1)
Coordinates:
  * techs                       (techs) object 16kB 'D0503100000000190_to_tra...
  * nodes                       (nodes) object 10kB 'D0503100000000190' ... '...
  * carriers                    (carriers) object 24B 'HQ_heat' ... 'electric...
  * timesteps                   (timesteps) datetime64[ns] 8B 2050-01-01
  * costs                       (costs) object 8B 'monetary'
Data variables: (12/20)
    flow_cap                    (nodes, techs, carriers) float64 58MB nan ......
    link_flow_cap               (techs) float64 16kB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    flow_out                    (nodes, techs, carriers, timesteps) float64 58MB ...
    flow_in                     (nodes, techs, carriers, timesteps) float64 58MB ...
    source_use                  (nodes, techs, timesteps) float64 19MB nan .....
    source_cap                  (nodes, techs) float64 19MB nan nan ... nan nan
    ...                          ...
    min_cost_optimisation       float64 8B 0.0
    capacity_factor             (nodes, techs, carriers, timesteps) float64 58MB ...
    systemwide_capacity_factor  (techs, carriers) float64 49kB 0.0 0.0 ... 0.0
    systemwide_levelised_cost   (techs, costs, carriers) float64 49kB nan ......
    total_levelised_cost        (costs, carriers) float64 24B nan nan nan
    unmet_sum                   (nodes, carriers, timesteps) float64 29kB nan...

In [143]:
# --- Calliope model results visualization ---

df_coords = model.inputs[["latitude", "longitude"]].to_dataframe().reset_index()
df_capacity = (
    model.results.flow_cap.where(model.inputs.base_tech == "transmission")
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)

# Define distribution and transmission dataframes for plotting
df_capacity_coords = pd.merge(df_coords, df_capacity, left_on="nodes", right_on="nodes").sort_values(by=['techs'])

# Extract link information from techs column (format: "node_from_to_node_to")
df_links = df_capacity_coords.copy()

# Split the techs column to get link_from and link_to
df_links[['link_from', 'link_to']] = df_links['techs'].str.rsplit('_to_', n=1, expand=True)

# Merge with df_coords twice to get both from and to coordinates
# First merge for "from" coordinates
df_links = df_links.merge(
    df_coords[['nodes', 'latitude', 'longitude']],
    left_on='link_from',
    right_on='nodes',
    how='left',
    suffixes=('', '_from')
)
df_links = df_links.rename(columns={'latitude': 'lat_from', 'longitude': 'lon_from'})

# Second merge for "to" coordinates
df_links = df_links.merge(
    df_coords[['nodes', 'latitude', 'longitude']],
    left_on='link_to',
    right_on='nodes',
    how='left',
    suffixes=('_temp', '_to')
)
df_links = df_links.rename(columns={'latitude': 'lat_to', 'longitude': 'lon_to'})

# Clean up duplicate columns
df_links = df_links.drop(columns=['nodes_temp', 'nodes_to'], errors='ignore')


# Create a Folium map centered on your data
center_lat = df_coords['latitude'].mean()
center_lon = df_coords['longitude'].mean()

map_fig = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=16,
    tiles='OpenStreetMap'
)


# Create FeatureGroups for different layers
demand_group = folium.FeatureGroup(name="Demand Nodes", show=True).add_to(map_fig)
supply_heat_group = folium.FeatureGroup(name="Supply Heat Nodes", show=True).add_to(map_fig)
supply_elec_group = folium.FeatureGroup(name="Supply Electricity Nodes", show=True).add_to(map_fig)
transmission_heat_group = folium.FeatureGroup(name="Heat Transmission Nodes", show=True).add_to(map_fig)
heat_link_group = folium.FeatureGroup(name="Heat Links", show=True).add_to(map_fig)
transmission_electricity_group = folium.FeatureGroup(name="Electricity Transmission Nodes", show=True).add_to(map_fig)
electricity_link_group = folium.FeatureGroup(name="Electricity Links", show=True).add_to(map_fig)

# Add lines for each link to the 'link_group' FeatureGroup
for idx, row in df_links.iterrows():
    if row['carriers'] == 'heat':
        color = 'green'
        target_group = heat_link_group
    else:
        color = 'blue'
        target_group = electricity_link_group
        
    folium.PolyLine(
        locations=[[row['lat_from'], row['lon_from']], [row['lat_to'], row['lon_to']]],
        color=color,
        weight=1,
        opacity=0.7,
        popup=f"<b>{row['techs']}</b><br>From: {row['link_from']}<br>To: {row['link_to']}<br>Capacity: {row['Flow capacity (kW)']} kW"
    ).add_to(target_group)


# Add node markers to their respective FeatureGroups
for idx, row in df_capacity_coords.iterrows():
    node_name = row['nodes']
    
    # Determine node type and styling
    if node_name.startswith('SH'):
        color = '#2ecc71' 
        radius = 1
        node_type = 'Supply heat'
        target_group = supply_heat_group
    elif node_name.startswith('SE'):
        color = "#2e38cc" 
        radius = 1
        node_type = 'Supply electricity'
        target_group = supply_elec_group
    elif node_name.startswith('D'):
        color = '#e74c3c'  
        radius = 1
        node_type = 'Demand'
        target_group = demand_group
    elif node_name.startswith('TH'):
        color = "#94d3ae" 
        radius = 1
        node_type = 'Transmission heat'
        target_group = transmission_heat_group
    else:  
        color = "#7076cc"  
        radius = 1
        node_type = 'Transmission electricity'
        target_group = transmission_electricity_group
    
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=radius,
        popup=f"<b>{row['nodes']}</b> ({node_type})<br>Capacity: {row['Flow capacity (kW)']} kW",
        color=color,
        fill=True,
        fillColor=color,
        fillOpacity=0.8,
        weight=2
    ).add_to(target_group) # Add to the correct group

# --- Add the LayerControl to the map ---
# This creates the toggle switch in the top-right corner
folium.LayerControl().add_to(map_fig)

# Display the map
map_fig.save("outputs/map_output.html")

ValueError: Columns must be same length as key

In [ ]:
# --- Bill of materials export ---

# For each item to be exported, find its name in inputs, merge with capacity data, and export to dataframe
tech_names = model.inputs.name.to_series().dropna()
tech_distances = model.inputs.distance.to_series().dropna()

total_flow_out = (
    model.results.flow_out
    .sum(dim=["nodes", "carriers", "timesteps"], min_count=1)
    .to_series()
    .dropna()
)

export_df = pd.DataFrame({
    'name': tech_names,
    'capacity_kw': total_flow_out,
    'distance_m': tech_distances*1000
})

final_export_df = export_df[export_df['capacity_kw'] > 0].sort_values(by='name')

# For each item in the export dataframe, multiply capacity by some environmental impact factor, and add environmental impact column

environmental_impact_factors = {
    "National grid import": 1,  # e.g. gCO2/kW
    "Geothermal heat extraction": 1,   
    "Heat transmission": 1,             
    "Electricity transmission": 1,      
    "Heat distribution": 1,             
    "Electricity distribution": 1,
    "Air-to-air heat pump": 1,      
}

final_export_df['environmental_impact_per_kW'] = final_export_df['name'].map(environmental_impact_factors)

# Sum total environmental impact across all items and output total environmental impact

final_export_df['environmental_impact'] = final_export_df['capacity_kw'] * final_export_df['environmental_impact_per_kW']
final_export_df = final_export_df.reset_index()

final_export_df.to_csv('outputs/bill_of_materials.csv', index=False)

final_export_df.head()

,techs,name,capacity_kw,distance_m,environmental_impact_per_kW,environmental_impact
